# 60 — Data quality and reconciliation gate
Results are appended to `gold.dq_results`. The notebook **fails the job** if any FAIL-severity check fails,
so a bad snapshot never goes unnoticed. Checks that cannot run (e.g. no JDBC secret) are reported as SKIPPED.

In [ ]:
%run ./_common

In [ ]:
results = []


def check(name, ok, detail="", severity="FAIL"):
    status = "PASS" if ok else severity
    results.append((name, status, str(detail)))
    print(f"[{status}] {name} {detail}")


asof_sk = int(AS_OF.replace("-", ""))
fact = spark.table(fq("gold", "fact_exposure_snapshot")).where(f"as_of_date_sk = {asof_sk}")
n_fact = fact.count()

In [ ]:
# --- Gold: grain, keys, referential integrity ---------------------------------------------------
check("fact_not_empty", n_fact > 0, f"rows={n_fact}")
check("fact_grain_unique", fact.select("exposure_sk").distinct().count() == n_fact, f"rows={n_fact}")
null_fks = fact.where("lc_sk IS NULL OR counterparty_sk IS NULL OR bank_sk IS NULL OR currency_sk IS NULL").count()
check("fact_fk_not_null", null_fks == 0, f"nulls={null_fks}")
check("fact_amounts_positive", fact.where("outstanding_exposure_local <= 0").count() == 0)
for fk, dim, key in [("lc_sk", "dim_lc", "lc_sk"), ("counterparty_sk", "dim_counterparty", "counterparty_sk"),
                     ("bank_sk", "dim_bank", "bank_sk"), ("currency_sk", "dim_currency", "currency_sk")]:
    orphans = fact.join(spark.table(fq("gold", dim)).select(F.col(key).alias("_k")), fact[fk] == F.col("_k"), "left_anti").count()
    check(f"ri_{fk}_to_{dim}", orphans == 0, f"orphans={orphans}")
unknown_cp = fact.where("counterparty_sk = -1").count()
check("unknown_counterparty_share_below_2pct", n_fact == 0 or unknown_cp / n_fact < 0.02, f"unknown={unknown_cp}/{n_fact}", "WARN")

# --- SCD2 integrity --------------------------------------------------------------------------------
hist = spark.table(fq("silver", "party_history"))
n_party = hist.select("party_id").distinct().count()
check("scd2_one_current_per_party", hist.where("is_current").count() == n_party, f"parties={n_party}")
w = Window.partitionBy("party_id").orderBy("eff_start_date")
gaps = hist.withColumn("_next", F.lead("eff_start_date").over(w)).where("_next IS NOT NULL AND _next <> eff_end_date").count()
check("scd2_no_gaps_or_overlaps", gaps == 0, f"bad_rows={gaps}")

In [ ]:
# --- Control total: fact vs independent recomputation from Silver ---------------------------------
asof = F.lit(AS_OF).cast("date")
ev = spark.table(fq("silver", "lc_exposure_event")).where(F.col("occurred_at") < F.date_add(asof, 1).cast("timestamp"))
w2 = Window.partitionBy("lc_id", "exposure_type").orderBy(F.col("occurred_at").desc(), F.col("event_id").desc())
recomputed = (ev.withColumn("_rn", F.row_number().over(w2)).where("_rn = 1 AND exposure_amount > 0")
              .join(spark.table(fq("silver", "letter_of_credit")).where(F.col("issue_date") <= asof).select("lc_id"), "lc_id")
              .join(spark.table(fq("silver", "currency_fx")).select("currency_code", "rate_to_usd"),
                    F.col("currency") == F.col("currency_code"))
              .agg(F.sum(F.col("exposure_amount") * F.col("rate_to_usd")).alias("usd")).first()["usd"])
fact_usd = fact.agg(F.sum("outstanding_exposure_usd")).first()[0]
recomputed = float(recomputed or 0)
fact_usd = float(fact_usd or 0)
variance = abs(fact_usd - recomputed) / recomputed if recomputed else (0 if fact_usd == 0 else 1)
check("control_total_variance_below_0.01pct", variance < 0.0001, f"fact={fact_usd:,.2f} recomputed={recomputed:,.2f} variance={variance:.6%}")

In [ ]:
# --- Bronze → Silver completeness: every Bronze key is in Silver or in quarantine ----------------------
quar = spark.table(fq("silver", "quarantine"))
for table in ["PARTY", "LETTER_OF_CREDIT", "LC_EXPOSURE_EVENT"]:
    key = TABLES[table]["pk"][0]
    b = spark.table(fq("bronze", table)).select(F.col(key).cast("string").alias("k")).distinct()
    s = spark.table(fq("silver", table)).select(F.col(key).cast("string").alias("k")).distinct()
    q = quar.where(F.col("source_table") == table).select(F.col("record_key").alias("k")).distinct()
    missing = b.join(s, "k", "left_anti").join(q, "k", "left_anti").count()
    check(f"bronze_keys_accounted_{table.lower()}", missing == 0, f"unaccounted={missing}")

# --- Landing → Bronze row counts -----------------------------------------------------------------------
for table in TABLES:
    try:
        landed = spark.read.parquet(f"{LANDING_ROOT}/sqldb/{table}").count()
    except Exception:
        landed = 0
    bronze = spark.table(fq("bronze", table)).count()
    check(f"landing_equals_bronze_{table.lower()}", landed == bronze, f"landing={landed} bronze={bronze}")

In [ ]:
# --- Source reconciliation (optional): rows in source up to the committed watermark == Bronze ----------
try:
    jdbc_url = dbutils.secrets.get(SECRET_SCOPE, "sql-jdbc-url")
    jdbc_user = dbutils.secrets.get(SECRET_SCOPE, "sql-admin-login")
    jdbc_pwd = dbutils.secrets.get(SECRET_SCOPE, "sql-admin-password")

    def src(query):
        return (spark.read.format("jdbc").option("url", jdbc_url).option("query", query)
                .option("user", jdbc_user).option("password", jdbc_pwd).load().first()[0])

    for table, key, typ in [("PARTY", "updated_at", "datetime2"), ("LETTER_OF_CREDIT", "updated_at", "datetime2"),
                            ("LC_EXPOSURE_EVENT", "event_id", "bigint")]:
        wm = src(f"SELECT last_watermark FROM ctl.watermark_control WHERE table_name = '{table}'")
        pk = TABLES[table]["pk"][0]
        src_n = src(f"SELECT COUNT(*) FROM tf.{table} WHERE {key} <= CAST('{wm}' AS {typ})")
        bronze_n = spark.table(fq("bronze", table)).select(pk).distinct().count()
        check(f"source_equals_bronze_{table.lower()}", int(src_n) == bronze_n, f"source={src_n} bronze_distinct={bronze_n}")
except Exception as exc:  # no scope / no network: report, do not fail
    results.append(("source_reconciliation", "SKIPPED", str(exc)[:200]))
    print("[SKIPPED] source reconciliation:", str(exc)[:120])

In [ ]:
out = spark.createDataFrame(results, "check_name string, status string, detail string") \
           .withColumn("run_ts", F.current_timestamp()).withColumn("as_of_date", F.lit(AS_OF).cast("date")) \
           .select("run_ts", "as_of_date", "check_name", "status", "detail")
out.write.format("delta").mode("append").saveAsTable(fq("gold", "dq_results"))
failed = [r for r in results if r[1] == "FAIL"]
if failed:
    raise Exception(f"{len(failed)} data-quality check(s) failed: {[r[0] for r in failed]}")
print(f"all {len(results)} checks passed or were non-blocking")